# 長い文書を分割して要約する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_summarization/long_document_summary.ipynb)

NLP クックブック 28-08 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

モデルに一度に入らない（あるいは入れると後半を読み飛ばす）長い文書を要約したい。分割して要約し、それをまとめ直す。

In [1]:
import io, re, urllib.request, zipfile
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat(prompt, max_new_tokens=160):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

text = aozora("https://www.aozora.gr.jp/cards/000035/files/1567_ruby_4948.zip")   # 走れメロス（約 1 万字）
sents = [s.strip() for s in re.split(r"(?<=[。！？」])(?![」。！？])|\n+", text) if s and s.strip()]

def make_chunks(sents, max_chars=2500):
    chunks, cur = [], ""
    for s in sents:
        if cur and len(cur) + len(s) > max_chars:
            chunks.append(cur); cur = ""
        cur += s
    return chunks + [cur]

chunks = make_chunks(sents)
print(len(text), "文字 →", len(chunks), "分割", [len(c) for c in chunks])

9913 文字 → 4 分割 [2488, 2490, 2495, 2333]


In [2]:
# 1. 分割ごとに要約する (map)。T4 なら 4 分割で 30 秒〜1 分ほど
partial = []
for i, chunk in enumerate(chunks, 1):
    s = chat(f"次は小説の一部です。この部分で起きた出来事を、登場人物の名前を入れて 2〜3 文で要約してください。要約だけを出力してください。\n\n{chunk}", max_new_tokens=140)
    partial.append(s)
    print(f"[{i}] {s}")

[1] メロスは激怒し、邪智暴虐の王を除かなければならぬと決意し、シラクスの市へとやって来きた。彼は村では牧人であり、妹の結婚式のために市に買い物に訪れていたのだが、市の寂しさと不安な雰囲気から、王政の乱れに気づく。メロスは老爺から、王が悪心を抱いていると疑われ、派手な暮らしをしている者には人質を差し出すよう命じ、逆らう者は十字架にかけられ殺されるという現状を知る。メロスは王に市を暴君の手から救うために短刀を差し出したが、王はメロスの言葉を信じず、メロスに三日間の猶予を与え、代わりに友人のセリヌンティウスを人質として差し出すよう要求する。
[2] セリヌンティウスが縄打たれた後、メロスは初夏にもかかわらず一睡もせずに十里を走り抜き、村に到着したのは翌日の午前だった。メロスは妹の結婚式の準備を知り、結婚式を翌日に延期するよう試みるが、婿の牧人に断られる。その後、祝宴に参加したメロスは結婚式を明日にするよう説得し、祝宴は夜通し行われた。しかし、メロスは出発を決意し、花嫁に別れを告げた後、眠りに落ち、翌朝に起床して王城へ向かう。雨に阻まれながらも、メロスは川の流れに阻まれ、友人を救うため、そして人の信実を示すために走り続けることになる。
[3] メロスが小説の中で経験した出来事は、濁流の水泳、山賊との遭遇と撃退、そして疲労による挫折である。しかし、その後メロスは再び立ち上がり、義務を果たすために前進することを決意する。彼は信頼に応えるため、名誉を守るために、肉体の疲労を克服し、希望を持って王城へ向かう。
[4] メロスは真実と友情を証明するためにシラクスへと走り、親友セリヌンティウスを救うために刑場へ間に合った。メロスは走る過程で様々な困難に直面したが、信頼と義務感により走り続け、最終的に友情と真実の力を示した。ディオニス王も彼らの姿に感銘を受け、信実の価値を認め、群衆とともに彼らを称えた。


In [3]:
# 2. 部分要約をつなげて、全体の要約を作る (reduce)
joined = "\n".join(f"{i}. {s}" for i, s in enumerate(partial, 1))
final = chat(f"次は、ある小説を前から順に分けて要約したものです。これらをまとめて、物語全体のあらすじを 4 文で書いてください。あらすじだけを出力してください。\n\n{joined}", max_new_tokens=220)
print(final)

メロスは邪智暴虐の王を除かなければならないと決意し、シラクスの市へとやってきた。彼は村では牧人であり、妹の結婚式のために市を訪れた際、王政の乱れに気づき、老爺から王が悪心を抱いていると知る。メロスは王に短刀を差し出して抵抗するが、王はメロスの言葉を信じず、友人のセリヌンティウスを人質として差し出すよう要求する。メロスはセリヌンティウスが縄打たれた後、十里を走り抜き、村に戻り結婚式を延期させる。その後、メロスは再び王城に向かい、様々な困難を乗り越えながら友情と真実を証明するために走り続ける。最終的にメロスは親友セリヌンティウスを救い、ディオニス王も彼らの信実を認め、群衆とともに彼らを称えた。


- 「分割して要約 → 要約を要約」は map-reduce 型と呼ばれる。分割ごとの処理は独立なので、並列化もできる。
- 部分要約の段階で落とした情報は、最終要約には戻らない。固有名詞や数値を残すよう、部分要約のプロンプトで指示する。
- 3B クラスのモデルでは、部分要約に話の取り違えや人名の誤りが混ざることがある（上の出力でも、短刀を持って王城に入り捕らえられたメロスが「王に短刀を差し出した」ことになり、最終要約では「短刀を差し出して抵抗する」に変わっている。妹の結婚式を翌日に繰り上げる場面は、部分要約の段階で「延期」と取り違え、最終要約にもそのまま残った）。誤りは最終要約にそのまま引き継がれるので、重要な用途ではより大きなモデルを使うか、この章の「要約が原文と食い違っていないか検査する」と組み合わせる。
- 前の部分の要約を次の部分の入力に加えていく方法（逐次的な洗練）だと、話のつながりは保ちやすいが、並列化はできず、誤りが後ろに伝わる。
- 長い入力をそのまま読めるモデルでも、文書が長くなるほど中ほどの内容を見落としやすい。分割は今でも有効な方法。